# Assignment 12: Predicting Hotel Booking Cancellations  
## Models: Naïve Bayes, Support Vector Machine (SVM), and Neural Network

**Objectives:**
- Understand how to use classification models (Naïve Bayes, SVM, Neural Networks) to predict hotel cancellations.
- Compare models in terms of accuracy, complexity, and business relevance.
- Interpret and communicate model results from a business perspective.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_12_bayes_svm_neural.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Hotel Bookings - Business Context
You work as a data analyst for a hospitality group that manages both **Resort** and **City Hotels**. One major challenge in operations is the unpredictability of **booking cancellations**, which affects staffing, inventory, and revenue planning.

You’ve been asked to use historical booking data to predict whether a future booking will be canceled. Your insights will help management plan more effectively.

Your tasks are to:
1. Build and evaluate three models: Naïve Bayes, SVM, and Neural Network.
2. Compare performance.
3. Recommend which model is best suited for the business needs.

### Key Use Cases
- Understand customer booking behavior
- Explore factors related to cancellations
- Segment guests based on booking characteristics
- Compare city vs. resort hotel performance




## Data Dictionary

This dataset contains booking information for two types of hotels: a **city hotel** and a **resort hotel**. Each record corresponds to a single booking and includes various details about the reservation, customer demographics, booking source, and whether the booking was canceled.

**Source**: [GitHub - TidyTuesday: Hotel Bookings](https://github.com/rfordatascience/tidytuesday/blob/master/data/2020/2020-02-11/readme.md)

| Variable | Type | Description |
|----------|------|-------------|
| `hotel` | character | Hotel type: City or Resort |
| `is_canceled` | integer | 1 = Canceled, 0 = Not Canceled |
| `lead_time` | integer | Days between booking and arrival |
| `arrival_date_year` | integer | Year of arrival |
| `arrival_date_month` | character | Month of arrival |
| `stays_in_weekend_nights` | integer | Nights stayed on weekends |
| `stays_in_week_nights` | integer | Nights stayed on weekdays |
| `adults` | integer | Number of adults |
| `children` | integer | Number of children |
| `babies` | integer | Number of babies |
| `meal` | character | Type of meal booked |
| `country` | character | Country code of origin |
| `market_segment` | character | Booking source (e.g., Direct, Online TA) |
| `distribution_channel` | character | Booking channel used |
| `is_repeated_guest` | integer | 1 = Repeated guest, 0 = New guest |
| `previous_cancellations` | integer | Past booking cancellations |
| `previous_bookings_not_canceled` | integer | Past bookings not canceled |
| `reserved_room_type` | character | Initially reserved room type |
| `assigned_room_type` | character | Room type assigned at check-in |
| `booking_changes` | integer | Number of booking modifications |
| `deposit_type` | character | Deposit type (No Deposit, Non-Refund, etc.) |
| `agent` | character | Agent ID who made the booking |
| `company` | character | Company ID (if booking through company) |
| `days_in_waiting_list` | integer | Days on the waiting list |
| `customer_type` | character | Booking type: Contract, Transient, etc. |
| `adr` | float | Average Daily Rate (price per night) |
| `required_car_parking_spaces` | integer | Requested parking spots |
| `total_of_special_requests` | integer | Number of special requests made |
| `reservation_status` | character | Final status (Canceled, No-Show, Check-Out) |
| `reservation_status_date` | date | Date of the last status update |

This dataset is ideal for classification, segmentation, and trend analysis exercises.


## 1. Load and Prepare the Hotel Booking Dataset

**Business framing:**  
Your hotel client wants to understand which bookings are most at risk of being canceled. But before modeling, your job is to prepare the data to ensure clean and reliable input.

### Do the following:
- Import data from the hotels dataset into a dataframe (in GitHub go to the DataSets folder and look for `hotels.csv`)
- Remove or impute missing values
- Encode categorical variables
- Create your `X` (features) and `y` (target = `is_canceled`)
- Split the data into training and test sets (70/30)

**Important:** Perform this split **before** any preprocessing or feature transformations.

### In Your Response:
1. How many total rows and columns are in the dataset?
2. What types of features (categorical, numerical) are included?
3. What steps did you take to clean or prepare the data?


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np

# 1. Import data from the hotels dataset
url = "https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/main/DataSets/hotels.csv"
df = pd.read_csv(url)

print(f"Original dataset shape: {df.shape}")

# Identify target variable
target = 'is_canceled'
y = df[target]
X = df.drop(columns=[target])

# Identify columns to drop due to high cardinality, many missing values, or data leakage
# 'reservation_status' directly tells if a booking was cancelled, causing data leakage.
# 'reservation_status_date' is also directly related.
# 'agent' and 'company' have many missing values and high cardinality for IDs.
columns_to_drop = ['agent', 'company', 'reservation_status', 'reservation_status_date']
X = X.drop(columns=columns_to_drop, errors='ignore')

# 5. Split the data into training and test sets (70/30)
# Important: Perform this split before any preprocessing or feature transformations.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

# Separate numerical and categorical columns for preprocessing
numerical_cols = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X_train.select_dtypes(include='object').columns.tolist()

# Preprocessing for numerical features: impute with median
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

# Preprocessing for categorical features: impute with mode and one-hot encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)) # Ensure dense output
])

# Create a preprocessor object using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ],
    remainder='passthrough'
)

# Fit the preprocessor on the training data and transform both training and test data
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Ensure processed data is dense, if it happens to be sparse (unlikely with sparse_output=False now)
if hasattr(X_train_processed, 'toarray'):
    X_train_processed = X_train_processed.toarray()
if hasattr(X_test_processed, 'toarray'):
    X_test_processed = X_test_processed.toarray()

# Get feature names after one-hot encoding for categorical columns
feature_names = preprocessor.get_feature_names_out()

print(f"Shape of X_train_processed before DataFrame conversion: {X_train_processed.shape}")
print(f"Number of feature_names: {len(feature_names)}")

# Convert back to DataFrame for better inspection and consistency for subsequent steps
X_train_processed_df = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
X_test_processed_df = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)

print(f"X_train_processed shape: {X_train_processed_df.shape}")
print(f"X_test_processed shape: {X_test_processed_df.shape}")

# Store the processed dataframes and target series for later use
# These will be used in subsequent steps for model training
# For SVM and Neural Network, StandardScaler will be applied on these processed dataframes
# For Naive Bayes, these processed dataframes can be used directly (e.g., GaussianNB expects continuous data).


Original dataset shape: (9404, 32)
X_train shape: (6582, 27)
X_test shape: (2822, 27)
y_train shape: (6582,)
y_test shape: (2822,)
Shape of X_train_processed before DataFrame conversion: (6582, 137)
Number of feature_names: 137
X_train_processed shape: (6582, 137)
X_test_processed shape: (2822, 137)


### ✍️ Your Response: 🔧
1. There are 119,390 rows, and 32 columns in the data set

2. The dataset contains both numerical and categorical data types.

3. In order to clean and prepare the data, I first analyzised the datset to identify data entries that will not be useful when anaylzing the data. I removed columns that had lots of missing data points. I also ensured that all data entries where in the corrent and usable format.

## 2. Build a Naïve Bayes Model

**Business framing:**  
Naïve Bayes is a quick, baseline model often used for early testing or simple classification problems.

### Do the following:
- Make sure to split your data first (see the previous step), then fit any text/vector preprocessing on training data only.
- Train a Naïve Bayes classifier on your training data
- Use it to predict on your test data
- Print a classification report and confusion matrix

**Note:** If you use a vectorizer (e.g., `CountVectorizer`), fit it on the training data only, then transform both training and test data.

### In Your Response:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. Where might this model be useful for the hotel (e.g. real-time alerts, operational decisions)?


In [4]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix

# Initialize and train the Naïve Bayes classifier
naive_bayes_model = GaussianNB()
naive_bayes_model.fit(X_train_processed_df, y_train)

# Make predictions on the test data
y_pred_nb = naive_bayes_model.predict(X_test_processed_df)

# Print classification report
print("Naïve Bayes Classification Report:")
print(classification_report(y_test, y_pred_nb))

# Print confusion matrix
print("\nNaïve Bayes Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_nb))


Naïve Bayes Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.37      0.54      2121
           1       0.34      0.99      0.51       701

    accuracy                           0.53      2822
   macro avg       0.67      0.68      0.53      2822
weighted avg       0.83      0.53      0.53      2822


Naïve Bayes Confusion Matrix:
[[ 790 1331]
 [   5  696]]


### ✍️ Your Response: 🔧
1. The model preformes with averages accuracy— this values was 0.53. I would like to see that model preform better than this, but all things considered, it is not the worst. The metric used to judge preformance is the canceled class. The goal of the model is to determine wether or not a booking is going to be cancled or not— therefor, this is the metric we used to judge preformance.

2. Because this model is used for predicting wether or not a booking is going to be cancled, the model may be useful for hotel managment. The managment could use the model to determine wether or not a booking is likey to be cancled— they can then look for ways to prevent customers from canceling the bookings. They can give incentives to customers or reach out to them prior to see what they can do to prevent the cancelation.

## 3. Build a Support Vector Machine (SVM) Model

**Business framing:**  
SVM can model more complex relationships and is useful when customer behavior patterns aren't linear or obvious.

### Do the following:
- Scale the data using `StandardScaler` to bring large numbers into a smaller range (Note: use a scaled training set, but fit the scaler only on X_train).
- Train an SVM classifier (use `linear` kernel)
- Make predictions and evaluate with classification metrics

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.   

### In Your Response:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. In what business situations could SVM provide better insights than simpler models?


In [5]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

# 1. Scale the data using StandardScaler
# Initialize the scaler
scaler = StandardScaler()

# Fit the scaler on the training data and transform both training and test data
X_train_scaled = scaler.fit_transform(X_train_processed_df)
X_test_scaled = scaler.transform(X_test_processed_df)

# Convert scaled arrays back to DataFrames for consistency and potential future use (optional, but good practice)
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train_processed_df.columns, index=X_train_processed_df.index)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test_processed_df.columns, index=X_test_processed_df.index)

print("Data scaled successfully.")
print(f"X_train_scaled shape: {X_train_scaled_df.shape}")
print(f"X_test_scaled shape: {X_test_scaled_df.shape}")

# 2. Train an SVM classifier (linear kernel)
# Using LinearSVC for large datasets with linear kernel, or SVC with kernel='linear' for more control
# For simplicity and potential performance, let's use SVC with 'linear' kernel first.
# Be mindful that SVC can be slow with large datasets.
print("\nTraining SVM model... This might take a while.")
svm_model = SVC(kernel='linear', random_state=42, verbose=True)
svm_model.fit(X_train_scaled_df, y_train)

print("SVM model trained.")

# 3. Make predictions on the test data
y_pred_svm = svm_model.predict(X_test_scaled_df)

# 4. Print classification report and confusion matrix
print("\nSVM Classification Report:")
print(classification_report(y_test, y_pred_svm))

print("\nSVM Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_svm))


Data scaled successfully.
X_train_scaled shape: (6582, 137)
X_test_scaled shape: (2822, 137)

Training SVM model... This might take a while.
[LibSVM]SVM model trained.

SVM Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.94      0.93      2121
           1       0.81      0.79      0.80       701

    accuracy                           0.90      2822
   macro avg       0.87      0.87      0.87      2822
weighted avg       0.90      0.90      0.90      2822


SVM Confusion Matrix:
[[1988  133]
 [ 145  556]]


### ✍️ Your Response: 🔧
1. The SVM model prefoms far better than the one from a few steps ago. The overall accuract of the SVM model was 0.90. This means that the model does a very good job at prediciting cancelations. The metric used to judge the preformace of the model is whether or not a booking is going to be cancled. In almost all cases, the model was able to correctly predict this.  

2. Due to the nature of SVMs, they are far supirior when it comes to data analyitics— especially when it comes to working with large and complex data sets. In a bussnies situation where we are trying to predict the out come of certian events, an SVM model can assist with out anaylis. They are far more accurate than simpilar models and will give the buisness the best shot at predicting the future. For exsample if a customer is likely to buy a product or not, etc...

## 4. Build a Neural Network Model

**Business framing:**  
Neural networks are flexible and powerful, though they are harder to explain. They may work well when subtle patterns exist in the data.

### Do the following:
- Build a MLPClassifier model using the neural_network package from sklearn
- Choose a simple architecture (e.g., 2 hidden layers)
- Use a true validation split from the training data, not the test set, for validation_data
- Evaluate accuracy and performance

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.  

### In Your Response:
1. How does this model compare to the others?
2. Would the business be comfortable using a “black box” model like this? Why or why not?


In [6]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix

# 1. Build a MLPClassifier model
# Choose a simple architecture (e.g., 2 hidden layers: 100 neurons in the first, 50 in the second)
# Use 'relu' activation and 'adam' solver, which are good defaults.
# max_iter is increased to ensure convergence, and random_state for reproducibility.
# validation_fraction is used to create a validation split from the training data.
print("\nTraining Neural Network model... This might take a while.")
mlp_model = MLPClassifier(
    hidden_layer_sizes=(100, 50), # 2 hidden layers with 100 and 50 neurons respectively
    max_iter=500,                  # Increased iterations for convergence
    activation='relu',             # Rectified Linear Unit activation function
    solver='adam',                 # Adam optimizer
    random_state=42,               # For reproducibility
    verbose=True,                  # To see training progress
    validation_fraction=0.1,       # Use 10% of training data for validation
    n_iter_no_change=10            # Stop if validation score doesn't improve for 10 consecutive epochs
)

# Train the model on the scaled training data
mlp_model.fit(X_train_scaled_df, y_train)

print("Neural Network model trained.")

# Make predictions on the test data
y_pred_mlp = mlp_model.predict(X_test_scaled_df)

# Evaluate accuracy and performance
print("\nNeural Network Classification Report:")
print(classification_report(y_test, y_pred_mlp))

print("\nNeural Network Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_mlp))



Training Neural Network model... This might take a while.
Iteration 1, loss = 0.41657653
Iteration 2, loss = 0.26636493
Iteration 3, loss = 0.22689642
Iteration 4, loss = 0.20576473
Iteration 5, loss = 0.19310788
Iteration 6, loss = 0.18261829
Iteration 7, loss = 0.17461783
Iteration 8, loss = 0.16679648
Iteration 9, loss = 0.15918123
Iteration 10, loss = 0.15530743
Iteration 11, loss = 0.14853667
Iteration 12, loss = 0.14437774
Iteration 13, loss = 0.13879801
Iteration 14, loss = 0.13543784
Iteration 15, loss = 0.13034151
Iteration 16, loss = 0.12759087
Iteration 17, loss = 0.12542392
Iteration 18, loss = 0.11937361
Iteration 19, loss = 0.11577388
Iteration 20, loss = 0.11245367
Iteration 21, loss = 0.11245892
Iteration 22, loss = 0.10832047
Iteration 23, loss = 0.10542376
Iteration 24, loss = 0.10177590
Iteration 25, loss = 0.10169513
Iteration 26, loss = 0.09797611
Iteration 27, loss = 0.09605788
Iteration 28, loss = 0.09498455
Iteration 29, loss = 0.09185390
Iteration 30, loss = 0

### ✍️ Your Response: 🔧
1. Compared to the Bayes model, this nural network model preformed much better. Just like the SVM model, it had an accuracy of 0.90. This is very high, and sugests that the model is very good at predicting the future.

2. I belive that in a lot of secerios, businesses would be comfortable using a model like this inorder to help them predict future events. In a lot a cases, predicitng the future can help businesses secure more sales, prevent canalations and ensure maximume profits. If a mdoel is predicitng with such high accracy as this one or the SVM, there is no reasona business would not consider using one.

## 5. Compare All Three Models

### Do the following:
- Print and compare the accuracy of Naïve Bayes, SVM, and Neural Network models
- Summarize which model performed best

### In Your Response:
1. Which model had the best overall accuracy, training time, interpretability, and ease of use.
2. Would you recommend this model for deployment, and why?


In [8]:
print(f"Naïve Bayes Model Accuracy: 53%")
print(f"SVM Model Accuracy: 90%")
print(f"Neural Network Model Accuracy: 90%")

# Although all three models' metrics are available in the previous outputs,
# for comparison of the 'canceled' class, let's also recall the F1-scores for Class 1 (canceled):
# Naïve Bayes Class 1 F1-score: 0.51
# SVM Class 1 F1-score: 0.80
# Neural Network Class 1 F1-score: 0.81

print(f"\nNaïve Bayes Model Class 1 (Canceled) F1-score: 0.51")
print(f"SVM Model Class 1 (Canceled) F1-score: 0.80")
print(f"Neural Network Model Class 1 (Canceled) F1-score: 0.81")

Naïve Bayes Model Accuracy: 53%
SVM Model Accuracy: 90%
Neural Network Model Accuracy: 90%

Naïve Bayes Model Class 1 (Canceled) F1-score: 0.51
SVM Model Class 1 (Canceled) F1-score: 0.80
Neural Network Model Class 1 (Canceled) F1-score: 0.81


### ✍️ Your Response: 🔧
1. The model that had the best overall accuracy was the SVM and Neural Network model. They both had an accuracy rating of 0.90. The model with the fastest training time was the Bayes model— the other two where slower. The Bayes model was the easist to use, followed by the SVM, and then the Neural network.
2. Taking this all into consideration, the model that I would recomend for someone to use would either be the SVM model or the Neural Network model. While they are more complicated to use, and take more time to train, their accracy is somthing that can not be underlooked.

## 6. Final Business Recommendation

### In Your Response:
1. In 100 words or less, write a short recommendation to hotel management based on your analysis.

Possible info to include:
- Which model do you recommend implementing?
- What business problem does it help solve?
- Are there any risks or limitations?
- What additional data might improve the results in the future?
2. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. I recommend that the hotel implement either the SVM or Neural Network model. These models have the most accuracy and will be the most usefull when it comes to making bussnies decisions.

1. 1.2 These models help some the business problem or predicitng what customers are most likely to cancel their hotel bookings. By being able to identify these customers, the hotel can take steps to prevent the cancelation through contact or incentives.

1. 1.3 There are some risks and limitaions that come with using these models. There can be false positives and false negitives. This can lead to lost revenue. Additionaly, if the data is not properly handled, there could be privicy concers for the hotel and customers.

1. 1.4 Additonal data like customer loalty, or holidays, etc... could give us a better insight into what customers are like. This goes for any data entry that gives us a closer look into the customer. Where are they from? What do they do for work? Do they have kids? etc...

2. This relates to my customized learning outcome because I want to anayliyze large datasets and be able to draw conclusions such as house price, future price, etc... Working with and creating models such as these gives me a deeper understanding into how I can predict these variables when it comes to working with datsets of my own.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [9]:
!jupyter nbconvert --to html "assignment_12_bayes_svm_neural_TettelbachIan.ipynb"

[NbConvertApp] Converting notebook assignment_12_bayes_svm_neural_TettelbachIan.ipynb to html
[NbConvertApp] Writing 341265 bytes to assignment_12_bayes_svm_neural_TettelbachIan.html
